# **Logit Lens: practice**

Hi, everyone! Welcome to the practice on Logit Lens. In this work you will implement the logit lens step by step — starting, as always, from the analysis of the architecture and finishing with plotting the results.

While working through the practice, you will:

* Analyse the architecture of GPT-2;
* Collect the logits from every component of the model that we need;
* Build a visualization of the obtained result;
* Get to know the NNsight library, which makes building logit lenses easier;


[The original post about LogitLiens](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)


Let's get started!

# **Part 1. A Logit Lens with your own hands**

In [ ]:
!pip install torchinfo==1.8.0 -q

In [ ]:
# Import libraries
from IPython.display import clear_output
from typing import List, Callable
import torch
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torchinfo import summary
import plotly.express as px
import plotly.io as pio


clear_output()

Let's start by loading the model.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval();

**Task 1. Call the ._modules attribute of the model. How many top-level modules did you get?**

In [ ]:
# Your code here

Let's look at how an input example is transformed inside GPT-2 step by step. To do that we will write a simple `input` and analyse the information about the model

In [ ]:
text = "Are cats good?"
encoded_input = tokenizer(text, return_tensors='pt')

input0 = torch.tensor(
    [[tokenizer.encode(text, add_special_tokens=True)]]
)
outputs = model(input0)

summary(model, input_data=input0)

**Tasks 2-4. Study the card you got above. Answer the questions:**

**1. How many parameters are there in the model in total?** 163,037,184 \
**2. How many transformer blocks are there in the model?** 12 \
**3. How many output values (tokens) does the model have?** 50257 \

Great, you have taken a rough look at the model. Now let's see how it works — we will push the input through the model, in order to get not only the information about the hidden states and the output logits, but also the generation of some text.

In [ ]:
output_ids = model.generate(**encoded_input, max_length=14, min_length=14, return_dict_in_generate=True, output_hidden_states=True)

# Decoding
output_text = tokenizer.decode(output_ids[0][0], skip_special_tokens=True)
output_text

**Note:** the first output tokens of the generation are exactly equal to the input.

**Task 5. Compare `output_ids['sequences']` and `encoded_input['input_ids']`. How many tokens following the input ones did the model generate?**

In [ ]:
# Your code here

In [ ]:
# Your code here

**Task 6. How many tokens did the model generate in total?**

In [ ]:
# Your code here

Now let's look at the hidden states. In the value of the dictionary we got there are 10 of them in total. However, the model has 12 transformer blocks and, on top of that, a hidden state is recorded after the `embedding` layer.

**Why did we get 10 hidden states for our input:** \

For GPT models the `hidden states` are returned for every generated token, and for each of them the model's `hidden states` are detailed. That is, since 10 new tokens were generated, we end up with 13 hidden states per token.  

In [ ]:
hs_tokens_cnt = len(output_ids['hidden_states'])
hs_f_each_tokens_cnt = len(output_ids['hidden_states'][0])

print(f'Number of tokens for which hidden states were generated {hs_tokens_cnt}\nNumber of hidden states per token: {hs_f_each_tokens_cnt}')

**Let's fix this once again:**

For every generated token there are 13 hidden states. You can convince yourself of this by running the code below with different tokens (change only the first line).

By the way, note that for token 198 (the value `\n`) we have only one set of hidden states. From this we have:

* Set of hidden states 0 for the input values: `tensor([ 8491, 11875,   922,    30])`
* Set of hidden states  1 for the token: `tensor([198])`
* Set of hidden states 2 for the token: `tensor([464])`
* Set of hidden states 3 for the token: `tensor([3280])`
* Set of hidden states 4 for the token: `tensor([318])`
* Set of hidden states 5 for the token: `tensor([3763])`
* Set of hidden states 6 for the token: `tensor([13])`
* Set of hidden states 7 for the token: `tensor([28997])`
* Set of hidden states 8 for the token: `tensor([389])`
* Set of hidden states 9 for the token: `tensor([922])`

In [ ]:
# Extract hidden states (last layer)
hidden_states = output_ids.hidden_states[0]  # Hidden states of the i-th token, a row in which the indices can be iterated over

last_hidden_state = hidden_states[0]  # The last state of the i-th token

# We extract the last layers for the projection onto the vocabulary
lm_head = model.lm_head  # linear layer to dict

# We convert the hidden states into logits manually
logits_from_hidden_states = lm_head(last_hidden_state)  # (batch_size, seq_length, vocab_size)

# We check that the shape is correct
print(f"Logits shape: {logits_from_hidden_states.shape}")  # Should be (batch_size, seq_length, vocab_size)

# We convert the logits into predictions (argmax over vocab size)
predicted_token_ids = torch.argmax(logits_from_hidden_states, dim=-1)

# We decode the predicted tokens
decoded_predictions = tokenizer.batch_decode(predicted_token_ids, skip_special_tokens=True)

# Results
print(f"Predicted tokens:\n{decoded_predictions}")

Now that we have understood what we collect for each token, we can collect the projections over all the tokens.

In [ ]:
lens_dict = {}
logits_dict = {}

for token_idx, token_hidden_states in enumerate(output_ids.hidden_states): # for the tuple of hidden states of every token
  lens_dict[token_idx] = []
  logits_dict[token_idx] = []

  for hidden_idx, hidden in enumerate(token_hidden_states): # for every hidden state inside

    logits = lm_head(hidden) # we extract the logits

    probs = torch.nn.functional.softmax(logits, dim=-1) # we extract the probabilities
    predicted =  torch.argmax(probs, dim=-1) # we extract the index of the predicted token

    proba =  torch.max(probs, dim=-1).values # we extract the probability of the predicted token
    logits_dict[token_idx].append(proba)

    decoded = tokenizer.batch_decode(predicted, skip_special_tokens=True)
    lens_dict[token_idx].append(decoded[0])

**Task 7. How many projections did you get in total?**

In [ ]:
# Your code here for every token (there are 10 of them) there are 13 hidden states = 10*13

Now, before visualizing the result, let's remove the first tokens and logits, since they reflect the input. On top of that, the first output contains several tokens, which is not convenient for displaying on a plot.

In [ ]:
logits_dict.pop(0)
lens_dict.pop(0)

### **Visualization of the result**

Let's move on to the visualization. All the collected logits and tokens can be seen as matrices, where the rows are the hidden layers and the columns are the tokens.

* The first row is the representation of the generated tokens at hidden layer 1; \
* The second row is the representation of the generated tokens at hidden layer 2;
* ...and so on...
* The last row is the representation of the generated tokens at the 13th hidden layer (the last encoder of the model).

The matrix, in turn, can be visualized as a heatmap. This is implemented below step by step.

In [ ]:
# Let's create a matrix for the probabilities
probs_matrix = []
tokens_matrix = []

# We iterate over all the tokens
for token_idx in range(1, len(lens_dict)+1):
    probs_matrix.append(logits_dict[token_idx])  # We collect the probabilities
    tokens_matrix.append(lens_dict[token_idx])  # We collect the tokens

# We convert both dictionaries into numpy for convenient work with heatmaps
probs_matrix = torch.stack([torch.tensor(i) for i in probs_matrix], dim=1).numpy()
tokens_matrix = np.stack([i for i in tokens_matrix])

# We visualize the probabilities with a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(probs_matrix, cmap='YlGnBu', annot=True, fmt=".2f", cbar=True)

# Let's set up the axis labels
plt.title("Logit Lens: Token Probabilities Across Layers")
plt.xlabel("Hidden States (Token Indexes)")
plt.ylabel("Layers")
plt.yticks(ticks=[i - 0.5 for i in range(1, len(lens_dict[1]) +1)], labels=[f'Layer {i}' for i in range(1, len(lens_dict[1]) + 1)], rotation=0)
plt.xticks(ticks=[i + 0.5 for i in range(0, len(lens_dict))], labels=[f"Token {i+1}" for i in range(0, len(lens_dict))])

# And let's show the map
plt.show()

With `matplotlib` we can visualize only the logits as a heatmap. It would be ideal to overlay the predicted words on every cell together with the logits.

This can be done with `plotly` — an interactive (but much heavier in terms of memory) library for visualization. Below I also give the code for it.

In [ ]:
output_token_ids = output_ids['sequences'][0][5:] # the output tokens, to write their indices along the x axis

In [ ]:
fig = px.imshow(
    probs_matrix, # the matrix of probabilities
    x=[str(i.item()) for i in output_token_ids], # the output tokens, to write their indices along the x axis
   # y=[f'L-r {i}' for i in list(range(len(tokens_matrix.T)))], # to display the names on the y axis (optional)
    color_continuous_scale=px.colors.diverging.RdYlBu_r, # changing the colour palette
    color_continuous_midpoint=0.50,  # the central point for the colour palette — for us it is 0.5, since we converted the logits into probabilities
    labels=dict(x="Out Tokens", y="Layers", color="Probability") # the names for the x and y axes and the colorbar
)

# putting the title on the plot

fig.update_layout(
    title='Logit Lens Visualization'
)

# putting the text on the cells

fig.update_traces(text=tokens_matrix.T, texttemplate="%{text}")

# visualizing the figure

fig.show()

Let's fix the conclusions from the plot:

1. The outputs of the logit lens at the first layer are approximately equal to the outputs of the logit lens at the last layer, with a shift;
2. The probabilities do not increase or decrease steadily across the layers. At the middle layers the model is the most confident in the tokens;

# **Part 2. NNsight**


[NNsight](https://nnsight.net/) is a library which, thanks to internal optimizations, lets you wrap HF models so that you can extract hidden states for further analysis quickly and simply.

**Advantages of the library:**

1. Speed of launching;
2. A convenient interface;
3. Plus clear tutorials with visualizations.

**Some disadvantages:**

As the authors write, the library is at an early stage. That is why there is still something to improve in it.

1. Not all models from HF can be loaded.
2. Readers of my blog have shared that when using it there can be errors with gradients in the graph.

However, for learning purposes the library is magnificent! Let's see how to build a lens with it.

In [ ]:
!pip install -U nnsight -q

In this library the models are loaded into wrapper classes with a syntax similar to transformers.

In [ ]:
# Loading GPT2
from nnsight import LanguageModel

model_NN = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)

Let's look at the architecture of the model.

In [ ]:
print(model_NN)

The architecture is almost the same. Except that there are the `transformer`, `generator` and `streamer` components. They are not described precisely in the documentation.

Now let's look at the implementation of the logit lens from the [tutorial](http://nnsight.net/notebooks/tutorials/logit_lens/) of the library authors. We will reproduce it with the same text that we used for the lens built by hand.

In [ ]:
prompt= "Are cats good?"
layers = model_NN.transformer.h
probs_layers = []

with model_NN.trace() as tracer:
    with tracer.invoke(prompt) as invoker:
        for layer_idx, layer in enumerate(layers):

            # The head transformation + layer normalization
            layer_output = model_NN.lm_head(model_NN.transformer.ln_f(layer.output[0]))

            # Applying softmax
            probs = torch.nn.functional.softmax(layer_output, dim=-1).save()
            probs_layers.append(probs)

probs = torch.cat([probs.value for probs in probs_layers])

# We take the maximum probabilities
max_probs, tokens = probs.max(dim=-1)

# We decode the ids into tokens
words = [[model_NN.tokenizer.decode(t.cpu()).encode("unicode_escape").decode() for t in layer_tokens]
    for layer_tokens in tokens]

# 'input_ids'
input_words = [model_NN.tokenizer.decode(t) for t in invoker.inputs[0][0]["input_ids"][0]]

## **Visualization**

In [ ]:
pio.renderers.default = "colab"


fig = px.imshow(
    max_probs.detach().cpu().numpy(),
    x=input_words,
    y=list(range(len(words))),
    color_continuous_scale=px.colors.diverging.RdYlBu_r,
    color_continuous_midpoint=0.50,
    text_auto=True,
    labels=dict(x="Input Tokens", y="Layers", color="Probability")
)

fig.update_layout(
    title='Logit Lens Visualization',
    xaxis_tickangle=0
)

fig.update_traces(text=words, texttemplate="%{text}")
fig.show()

Note that the results do not match. And this highlights the problem with using wrapper libraries — you do not fully control the result and you cannot see every component that leads to it.


Thank you for your work!